In [2]:
"""
Knowledge Distillation — Script 4: Self-Distillation (Best Config Only)
========================================================================
Model   : ResNet-50 (modified for CIFAR-10)
Dataset : CIFAR-10
Method  : CS-KD (Class-wise Self-Knowledge Distillation) — tau=4.0, epochs=10

CS-KD enforces consistent predictions across two augmented views of the SAME
CLASS. Given two images from the same class (xa, xb):
  L_CSKD = CE(y, p_a) + tau^2 * KL(p_a.detach() || p_b)

Output : __4__KD_self_distillation_cskd_best.json
"""

import os, sys, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch {torch.__version__}", flush=True)
print(f"[init] Device  : {DEVICE}", flush=True)
if DEVICE.type == "cuda":
    print(f"[init] GPU     : {torch.cuda.get_device_name(0)}", flush=True)

# ── Config ─────────────────────────────────────────────────────────────────────
BASELINE_MODEL_PATH   = "../../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../../__2__baseline_metrics.json"
OUTPUT_JSON           = "__4__KD_self_distillation_cskd_best.json"

BATCH_SIZE  = 128 if DEVICE.type == "cuda" else 64
NUM_WORKERS = 0
PIN_MEMORY  = False
NUM_CLASSES = 10
KD_EPOCHS   = 10
USE_AMP     = (DEVICE.type == "cuda")

# Best config (from prior sweep)
CSKD_TAU = 4.0

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] AMP     : {USE_AMP}", flush=True)
print(f"[init] Config  : CS-KD | tau={CSKD_TAU} | epochs={KD_EPOCHS}", flush=True)


# ── Model builder ──────────────────────────────────────────────────────────────
def build_resnet50_cifar(num_classes: int = 10) -> nn.Module:
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_baseline(path: str) -> nn.Module:
    print(f"[load] Loading baseline from {path} ...", flush=True)
    model = build_resnet50_cifar(NUM_CLASSES)
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval()
    print("[load] OK", flush=True)
    return model


# ── Data ────────────────────────────────────────────────────────────────────────
def get_train_loader():
    transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../../data", train=True,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=512, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)


# ── Helpers ─────────────────────────────────────────────────────────────────────
def model_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)

def param_count_nonzero(model: nn.Module) -> int:
    return int(sum((p != 0).sum().item() for p in model.parameters()))

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

@torch.no_grad()
def evaluate_standard(model: nn.Module, loader: DataLoader) -> dict:
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(DEVICE, non_blocking=True)
        out    = model(inputs)
        logits = out[0] if isinstance(out, tuple) else out
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_inference_detailed(model: nn.Module, device: torch.device) -> dict:
    model = copy.deepcopy(model).eval().to(device)
    is_gpu = device.type == "cuda"

    # ── Single image ──────────────────────────────────────────
    dummy_single = torch.randn(1, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(50): model(dummy_single)
    if is_gpu: torch.cuda.synchronize()

    times = []
    with torch.no_grad():
        for _ in range(500):
            if is_gpu: torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single)
            if is_gpu: torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    single_ms = float(np.mean(times) * 1000)

    # ── Batch 128 ─────────────────────────────────────────────
    dummy_batch = torch.randn(128, 3, 32, 32, device=device)
    if is_gpu:
        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)
        with torch.no_grad():
            for _ in range(10): model(dummy_batch)
        torch.cuda.synchronize()
        batch_times = []
        with torch.no_grad():
            for _ in range(100):
                start_ev.record()
                model(dummy_batch)
                end_ev.record()
                torch.cuda.synchronize()
                batch_times.append(start_ev.elapsed_time(end_ev))
        batch_ms = float(np.mean(batch_times))
    else:
        with torch.no_grad():
            for _ in range(5): model(dummy_batch)
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(20): model(dummy_batch)
        batch_ms = (time.perf_counter() - t0) / 20 * 1000

    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms, 4),
        "per_img_gpu_ms"     : round(batch_ms / 128, 4),
        "throughput_imgs_sec": round(128 / (batch_ms / 1000), 1),
        "timing_method"      : "CUDA events + torch.cuda.synchronize()",
    }

def compute_flops(model: nn.Module) -> float:
    from thop import profile as thop_profile
    m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(1, 3, 32, 32)
    try:
        macs, _ = thop_profile(m, inputs=(dummy,), verbose=False)
        return round(float(macs * 2 / 1e9), 4)
    except Exception as e:
        print(f"      [flops] WARNING: thop failed ({e})", flush=True)
        return None


# ══════════════════════════════════════════════════════════════════════════════
# CS-KD training loop
# ══════════════════════════════════════════════════════════════════════════════
def run_cskd(baseline_model, train_loader, test_loader, baseline_acc,
             tau: float = 4.0, n_epochs: int = KD_EPOCHS) -> dict:
    """
    CS-KD: within each batch, pair samples of the same class and enforce
    consistent predictions. Uses a batch-level same-class grouping trick.
    """
    print(f"\n  ┌─ [CS-KD / tau={tau} / epochs={n_epochs}]", flush=True)

    try:
        model = copy.deepcopy(baseline_model).to(DEVICE)

        optimizer = torch.optim.SGD(model.parameters(), lr=0.01,
                                    momentum=0.9, weight_decay=1e-4, nesterov=True)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
        scaler    = torch.amp.GradScaler(enabled=USE_AMP)

        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        epoch_train_acc = []
        t_start = time.perf_counter()

        for epoch in range(n_epochs):
            model.train()
            correct = total = 0
            t_ep = time.perf_counter()

            for batch_idx, (inputs, targets) in enumerate(train_loader):
                inputs  = inputs.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    logits  = model(inputs)
                    ce_loss = F.cross_entropy(logits, targets)

                    # CS-KD: for each class, enforce consistency between pairs
                    kd_loss  = torch.tensor(0.0, device=DEVICE)
                    num_pairs = 0
                    for cls in targets.unique():
                        idx = (targets == cls).nonzero(as_tuple=True)[0]
                        if idx.numel() < 2:
                            continue
                        n_pairs = idx.numel() // 2
                        idx_a   = idx[:n_pairs]
                        idx_b   = idx[n_pairs:n_pairs * 2]
                        p_a     = F.log_softmax(logits[idx_a] / tau, dim=1)
                        p_b     = F.softmax(logits[idx_b].detach() / tau, dim=1)
                        kd_loss = kd_loss + F.kl_div(p_a, p_b, reduction="batchmean") * (tau ** 2)
                        num_pairs += n_pairs

                    if num_pairs > 0:
                        kd_loss = kd_loss / num_pairs

                    loss = ce_loss + kd_loss

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                correct += (logits.detach().argmax(1) == targets).sum().item()
                total   += targets.size(0)

                if (batch_idx + 1) % 100 == 0:
                    elapsed = time.perf_counter() - t_ep
                    done    = batch_idx + 1
                    eta     = elapsed / done * (len(train_loader) - done)
                    print(f"      [epoch {epoch+1}/{n_epochs}] "
                          f"batch {done}/{len(train_loader)}  "
                          f"acc={correct/total:.4f}  ETA={eta:.0f}s", flush=True)

            scheduler.step()
            acc = correct / total
            epoch_train_acc.append(round(acc, 6))
            ep_time   = time.perf_counter() - t_ep
            remaining = n_epochs - (epoch + 1)
            print(f"      [epoch {epoch+1}/{n_epochs}] DONE  "
                  f"acc={acc:.4f}  time={ep_time:.1f}s  "
                  f"ETA_remaining={ep_time * remaining:.0f}s", flush=True)

        sync()
        train_time_s = time.perf_counter() - t_start
        peak_gpu_mb  = (round(torch.cuda.max_memory_allocated() / 1024 ** 2, 2)
                        if DEVICE.type == "cuda" else None)

        # ── Evaluation ────────────────────────────────────────────────────────
        print("      [eval] Classification metrics ...", flush=True)
        metrics = evaluate_standard(model, test_loader)

        print("      [eval] Inference timing ...", flush=True)
        inference_info = measure_inference_detailed(model, DEVICE)

        print("      [eval] FLOPs ...", flush=True)
        flops_g = compute_flops(model)

        size_mb      = model_size_mb(model)
        params_nz    = param_count_nonzero(model)
        acc_drop     = baseline_acc - metrics["accuracy"]

        print(f"  └─ Acc={metrics['accuracy']:.4f}  Drop={acc_drop:+.4f}  "
              f"Train={train_time_s:.1f}s", flush=True)

        return {
            "sweep"             : "CS-KD",
            "variant"           : "cskd",
            "temperature"       : tau,
            "epochs"            : n_epochs,
            "train_device"      : str(DEVICE),
            "use_amp"           : USE_AMP,
            "train_time_s"      : round(train_time_s, 2),
            "epoch_train_acc"   : epoch_train_acc,
            "accuracy"          : round(metrics["accuracy"],  6),
            "precision"         : round(metrics["precision"], 6),
            "recall"            : round(metrics["recall"],    6),
            "f1"                : round(metrics["f1"],        6),
            "accuracy_drop"     : round(acc_drop, 6),
            "params_nonzero"    : params_nz,
            "size_mb"           : round(size_mb, 4),
            "flops_G"           : flops_g,
            "inference_ms"      : inference_info,
            "peak_gpu_memory_mb": peak_gpu_mb,
            "description"       : f"CS-KD: tau={tau}, epochs={n_epochs}",
            "status"            : "ok",
        }

    except Exception as e:
        import traceback
        print(f"  └─ FAILED: {e}", flush=True)
        traceback.print_exc()
        return {
            "variant": "cskd", "temperature": tau,
            "status": "failed", "error": str(e),
            "accuracy": None, "accuracy_drop": None,
        }


# ── Main ────────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  Self-Distillation — CS-KD (Best Config)")
    print(f"  tau={CSKD_TAU}  |  epochs={KD_EPOCHS}  |  device={DEVICE}  |  batch={BATCH_SIZE}")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline acc : {baseline_acc:.4f}\n", flush=True)

    baseline_model   = load_baseline(BASELINE_MODEL_PATH)
    baseline_size_mb = model_size_mb(baseline_model)
    baseline_params  = param_count_nonzero(baseline_model)
    print(f"  Baseline — Size: {baseline_size_mb:.2f} MB  "
          f"Non-zero params: {baseline_params:,}\n", flush=True)

    train_loader = get_train_loader()
    test_loader  = get_test_loader()

    # ── Single CS-KD run (best config) ────────────────────────────────────────
    result = run_cskd(
        baseline_model, train_loader, test_loader,
        baseline_acc, tau=CSKD_TAU, n_epochs=KD_EPOCHS,
    )

    report = {
        "method"      : "self_distillation_cskd_best",
        "description" : "CS-KD best config: tau=4.0, epochs=10 (no external teacher)",
        "model_arch"  : "ResNet-50 (CIFAR-10 modified)",
        "dataset"     : "CIFAR-10",
        "train_device": str(DEVICE),
        "use_amp"     : USE_AMP,
        "kd_epochs"   : KD_EPOCHS,
        "baseline"    : baseline,
        "baseline_model_info": {
            "size_mb"     : round(baseline_size_mb, 4),
            "params_nonzero": baseline_params,
        },
        "result": result,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n  ✓ Saved → {OUTPUT_JSON}")
    if result.get("accuracy") is not None:
        print(f"  Acc={result['accuracy']:.4f}  |  Drop={result['accuracy_drop']:+.4f}  |  "
              f"F1={result['f1']:.4f}  |  "
              f"Throughput={result['inference_ms']['throughput_imgs_sec']:.0f} img/s  |  "
              f"FLOPs={result['flops_G']} G")


if __name__ == "__main__":
    main()

[init] PyTorch 2.12.0.dev20260324+cu128
[init] Device  : cuda
[init] GPU     : NVIDIA GeForce RTX 5070 Laptop GPU
[init] AMP     : True
[init] Config  : CS-KD | tau=4.0 | epochs=10

  Self-Distillation — CS-KD (Best Config)
  tau=4.0  |  epochs=10  |  device=cuda  |  batch=128

  Baseline acc : 0.9320

[load] Loading baseline from ../../__2__baseline_resnet50_cifar10.pth ...
[load] OK
  Baseline — Size: 90.03 MB  Non-zero params: 23,520,842


  ┌─ [CS-KD / tau=4.0 / epochs=10]
      [epoch 1/10] batch 100/391  acc=0.9664  ETA=46s
      [epoch 1/10] batch 200/391  acc=0.9536  ETA=37s
      [epoch 1/10] batch 300/391  acc=0.9484  ETA=20s
      [epoch 1/10] DONE  acc=0.9459  time=88.3s  ETA_remaining=794s
      [epoch 2/10] batch 100/391  acc=0.9413  ETA=65s
      [epoch 2/10] batch 200/391  acc=0.9423  ETA=31s
      [epoch 2/10] batch 300/391  acc=0.9423  ETA=13s
      [epoch 2/10] DONE  acc=0.9422  time=53.3s  ETA_remaining=427s
      [epoch 3/10] batch 100/391  acc=0.9458  ETA=32s
    